# NVIDIA DLI: Triton Inference Server for Recommender Systems

This notebook is adapted from the NVIDIA Deep Learning Institute (DLI) course on deploying
recommender systems using NVIDIA Triton Inference Server. It covers exporting a TensorFlow
model, deploying it to Triton, sending inference requests, and monitoring server metrics.

**Requirements:**
- `tritonclient[http]` and `tritonclient[grpc]` -- Triton client libraries
- `nvtabular` -- NVIDIA NVTabular for feature engineering and model export
- `tensorflow` -- TensorFlow for model loading
- `cudf` (RAPIDS) -- GPU-accelerated DataFrames
- `numpy`, `pandas` -- standard data manipulation

**Note:** This notebook was originally designed to run on the NVIDIA DLI platform with
a pre-configured Triton Inference Server instance. Some cells (Triton server communication,
model loading/unloading, metrics) require a running Triton server and will not execute
in a standalone environment without additional setup.

# Triton for Recommender Systems

The [Triton Inference Server](https://github.com/triton-inference-server/server/blob/main/README.md#documentation) allows us to deploy our model to the web regardless of cloud provider, and it supports a number of different machine learning frameworks such as TensorFlow and PyTorch.

## Objectives
* Learn how to deploy a model to Triton
  * [1. Deploy TensorFlow Model to Triton Inference Server](#1.-Deploy-TensorFlow-Model-to-Triton-Inference-Server)
      * [1.1 Export a Model](#1.1-Export-a-Model)
      * [1.2 Review exported files](#1.2-Review-exported-files)
      * [1.3 Loading a Model](#1.3-Loading-a-Model)
  * [2. Sent requests for predictions](#2.-Sent-requests-for-predictions)
* Learn how to record deployment metrics
  * [3. Server Metrics](#3.-Server-Metrics)

## 1. Deploy TensorFlow Model to Triton Inference Server

Our Triton server has already been launched to the web and is ready to make requests. First, we need to export the saved TensorFlow model from Lab 2 and generate the config file for Triton Inference Server. NVTabular provides an easy-to-use function, which manages both tasks.

### 1.1 Export a Model

---
## Environment Setup

The cell below contains the consolidated installation commands for all required
dependencies. It is commented out by default. Uncomment and run if you need to
install packages in your environment (e.g., Colab or a fresh VM).

**For RAPIDS (cudf):** Follow the official RAPIDS installation guide at
https://rapids.ai/start.html for your CUDA version. The Colab-specific RAPIDS
installer script is shown below but may need adjustments for your platform.

In [ ]:
# =============================================================================
# INSTALLATION CELL (uncomment the lines you need)
# =============================================================================

# -- Triton client libraries --
# !pip install tritonclient[http] tritonclient[grpc]

# -- NVTabular (NVIDIA feature engineering + Triton export utilities) --
# !pip install nvtabular

# -- RAPIDS / cuDF (GPU DataFrames) --
# For Colab, the RAPIDS team provides helper scripts:
#   !git clone https://github.com/rapidsai/rapidsai-csp-utils.git
#   !python rapidsai-csp-utils/colab/env-check.py
#   !bash rapidsai-csp-utils/colab/update_gcc.sh
#   import condacolab; condacolab.install()
#   !python rapidsai-csp-utils/colab/install_rapids.py stable
# For other environments, see: https://rapids.ai/start.html

# -- Standard ML libraries (usually pre-installed) --
# !pip install tensorflow numpy pandas

---
## Imports

In [ ]:
# Standard library
import os
import sys
import argparse
from time import time

# Data manipulation
import numpy as np
import pandas as pd

# GPU DataFrames (RAPIDS)
import cudf

# TensorFlow
import tensorflow as tf

# Triton client libraries
import tritonhttpclient
import tritonclient.grpc as grpcclient

# NVTabular
import nvtabular
import nvtabular.inference as nvi
import nvtabular.inference.triton as nvt_triton

---
## Dataset Configuration

Set the paths below to point to your local data directory.
The model zip file (`task_2_model.zip`) should contain the saved TensorFlow model
from the previous lab in the DLI course.

In [ ]:
# ---- Configure these paths for your environment ----
# If running on the DLI platform, these are pre-configured.
# If running locally, update DATA_DIR to point to your data folder.

DATA_DIR = "./data"  # Directory containing task_2_model.zip and task_2_wide_and_deep.csv
MODEL_ZIP_PATH = os.path.join(DATA_DIR, "task_2_model.zip")

# Original path (Google Colab / Google Drive):
# MODEL_ZIP_PATH = "/content/drive/MyDrive/Recommender_Intelligent_Systems/RecommenderSystems/data/task_2_model.zip"

Let's unzip the model that we saved as a zip file in the previous notebook, and then load it to be able to use it in the NVTabular `export_tensorflow_model()` function below.

In [ ]:
!unzip -o "$MODEL_ZIP_PATH"

Next, we will load the TensorFlow model.

In [ ]:
model = tf.keras.models.load_model('task2_model')

Since we will need the output name of the last layer to make predictions later, let's print them out using `model.output_names`.

In [ ]:
model.output_names

We can export the model to `model_repository`. This folder is shared between the docker container for the jupyter notebook and the docker container that runs Triton Inference Server. Therefore, Triton will have access to the model files.

In [ ]:
# Generate the TF saved model for Triton
from nvtabular.inference.triton.ensemble import export_tensorflow_model

tf_config = export_tensorflow_model(model, "wnd_tf", "model_repository/wnd_tf", version=1)

**Note:** In the original DLI notebook, a kernel restart was performed here to free GPU
memory after exporting the model. If you are running this locally and encounter memory
issues, restart the kernel manually at this point and re-run the imports cell above before
continuing.

### 1.2 Review exported files

Let's look at the files `export_tensorflow_model` created. Triton expects [a specific directory structure](https://github.com/triton-inference-server/server/blob/main/docs/model_repository.md) for our models. The folder `/model_repository` is shared with our server, and it expects the following format:

```
<model_repository_path>/
  <model-name>/
    [config.pbtxt]
    <version-name>/
      [model.savedmodel]/
        <tensorflow_saved_model_files>/
          ...
```

In [ ]:
!tree model_repository

Let's look at the generated config file. It defines the input columns with datatype and dimensions and the output layer. Manually creating this config file can be complicated and NVTabular provides an easy function with `export_tensorflow_model` to deploy TensorFlow model to Triton.

Triton needs a [config file](https://github.com/triton-inference-server/server/blob/main/docs/model_configuration.md) to understand how to interpret the model. Our `export_tensorflow_model` method is automatically creating the config file and the required folder structure for us, so that we do not need to create it manually.

The config file needs the following information:
* name: The name of our model. Must be the same name as the parent folder.
* platform: The type of framework serving the model.
* input: The input our model expects.
  * `name`: Should correspond with the model input name.
  * `data_type`: Should correspond to the input's data type.
  * `dims`: The dimensions of the *request* for the input, as in the dimensions of the data the user passes to us.
  * `reshape`: How to reshape the data from the client to pass it to our model. In this case, the minimum dims from the client is `[1]`, but like Keras, Triton appends a dimension for batching. If our model expects `[batch_size]` as a dimension, we can reshape our data to `[]` (empty brackets) to account for that.
* output: The output parameters of our model.
  * `name`: Should correspond with the model output name. In this case, we're using the name automatically assigned by TensorFlow.
  * `data_type`: Should correspond to the output's data type.
  * `dims`: The dimensions of the output.

In [ ]:
!cat model_repository/wnd_tf/config.pbtxt

### 1.3 Loading a Model

Now, we can communicate with the Triton Inference Server and sent the request to load the model. We can verify this by using [curl](https://curl.haxx.se/) to make a `GET` request.

In [ ]:
!curl -i triton:8000/v2/health/ready

Next, let's build a client to connect to our server. This [InferenceServerClient](https://github.com/triton-inference-server/client) object is what we'll be using to talk to Triton.

In [ ]:
try:
    triton_client = tritonhttpclient.InferenceServerClient(url="triton:8000", verbose=True)
    print("client created.")
except Exception as e:
    print("channel creation failed: " + str(e))

We can verify that our server is ready to go by using [is_server_live](https://github.com/triton-inference-server/client/blob/12d8a2a7318ccb4a367a09a42b80feba53f3944a/src/python/library/tritonclient/grpc/__init__.py#L259). [get_model_repository_index](https://github.com/triton-inference-server/client/blob/12d8a2a7318ccb4a367a09a42b80feba53f3944a/src/python/library/tritonclient/grpc/__init__.py#L555) will also show what folders are in Triton's model repository.

In [ ]:
triton_client.is_server_live()

In [ ]:
triton_client.get_model_repository_index()

Now that everything is configured, let's get the model loaded! First, we'll create a version for our model. By default, Triton loads the version in the first listed folder, so we'll use `1` for our version number.

Finally, we'll copy our model into the server.

We've set Triton's [Model Control Mode](https://github.com/triton-inference-server/server/blob/main/docs/model_management.md#model-control-mode-explicit) to `EXPLICIT`, meaning, it's not going to automatically pick up the model placed in its directory. This is done on line 13 of our `docker-compose.yml` file in the [previous lab](3-02_docker.ipynb). We could have used [POLL](https://github.com/triton-inference-server/server/blob/main/docs/model_management.md) in order to do this, but it's not immediate when checking for changes.

In order to load our model, we'll use [load_model](https://github.com/triton-inference-server/client/blob/12d8a2a7318ccb4a367a09a42b80feba53f3944a/src/python/library/tritonclient/grpc/__init__.py#L601). When needed, we can use [unload_model](https://github.com/triton-inference-server/client/blob/12d8a2a7318ccb4a367a09a42b80feba53f3944a/src/python/library/tritonclient/grpc/__init__.py#L634) when we want to remove it from the Triton server.

In [ ]:
model_name = "wnd_tf"

try:
    triton_client.load_model(model_name=model_name)
    print(f"Model '{model_name}' loaded successfully.")
except Exception as e:
    print(f"Failed to load model '{model_name}': {e}")
    print("Ensure the Triton server is running and the model repository is correctly configured.")
    raise

Now that the model is loaded, we can use [get_model_metadata](https://github.com/triton-inference-server/client/blob/12d8a2a7318ccb4a367a09a42b80feba53f3944a/src/python/library/tritonclient/grpc/__init__.py#L429) to see our model's inputs and outputs.

In [ ]:
try:
    triton_client.get_model_metadata(model_name=model_name)
except Exception as e:
    print(f"Failed to retrieve model metadata: {e}")
    print("Ensure the model is loaded on the Triton server before requesting metadata.")
    raise

Ok, time to shine! Let's make a request to our server!

### 2. Sent requests for predictions

We can use [InferInput](https://github.com/triton-inference-server/client/blob/12d8a2a7318ccb4a367a09a42b80feba53f3944a/src/python/library/tritonclient/grpc/__init__.py#L1449) to describe the tensors we'll be sending to the server. It needs the name of the input, the shape of the tensor we'll be passing to the server, and its datatype.

Then, we can use [set_data_from_numpy](https://github.com/triton-inference-server/client/blob/12d8a2a7318ccb4a367a09a42b80feba53f3944a/src/python/library/tritonclient/grpc/__init__.py#L1513) to pass it a NumPy array.

We'll use some fake data for now. The first row of our batch will have all `1`s and the second will have all `2`s.

In [ ]:
inputs = []
outputs = []
batch_size = 2
inputs.append(tritonhttpclient.InferInput("user_index", [batch_size, 1], "INT64"))
inputs.append(tritonhttpclient.InferInput("item_index", [batch_size, 1], "INT64"))
inputs.append(tritonhttpclient.InferInput("brand_index", [batch_size, 1], "INT64"))
inputs.append(tritonhttpclient.InferInput("price_filled", [batch_size, 1], "FP32"))
inputs.append(tritonhttpclient.InferInput("salesRank_Electronics", [batch_size, 1], "FP32"))
inputs.append(tritonhttpclient.InferInput("category_0_2_index", [batch_size, 1], "INT32"))
inputs.append(tritonhttpclient.InferInput("category_1_2_index", [batch_size, 1], "INT32"))

inputs[0].set_data_from_numpy(np.array([[1], [2]], dtype=np.int64))
inputs[1].set_data_from_numpy(np.array([[1], [2]], dtype=np.int64))
inputs[2].set_data_from_numpy(np.array([[1], [2]], dtype=np.int64))
inputs[3].set_data_from_numpy(np.array([[1.0], [2.0]], dtype=np.float32))
inputs[4].set_data_from_numpy(np.array([[1.0], [2.0]], dtype=np.float32))
inputs[5].set_data_from_numpy(np.array([[1], [2]], dtype=np.int32))
inputs[6].set_data_from_numpy(np.array([[1], [2]], dtype=np.int32))

# NOTE: The output name "tf.__operators__.add" is auto-generated by TensorFlow based on
# the final operation in the model graph. This name may vary depending on your TensorFlow
# version or model architecture. Check model.output_names (cell above) or the Triton model
# metadata to confirm the correct output name for your model.
outputs.append(
    tritonhttpclient.InferRequestedOutput("tf.__operators__.add", binary_data=False)
)

try:
    results = triton_client.infer(model_name, inputs, outputs=outputs).get_response()
except Exception as e:
    print(f"Inference request failed: {e}")
    print("Ensure the Triton server is running and the model is loaded (see cells above).")
    raise

We'll get a bunch of data returned from our response, but the important one is the `"data"` at the very end. That's our prediction from our model!

In [ ]:
results["outputs"][0]["data"]

This seems like a lot of work for only two predictions. Can we give it something meatier? We have loaded in the data from our previous labs. Let's try running our validation data from lab2 through the server.

In [ ]:
ratings = pd.read_csv(os.path.join(DATA_DIR, "task_2_wide_and_deep.csv"))
ratings = ratings[ratings["valid"]]
ratings.head()

Let's try to be a little more efficient with our code this time. We'll use a `for` loop to construct our inputs.

In [ ]:
columns = [
    ('user_index', "INT64"),
    ('item_index', "INT64"),
    ('brand_index', "INT64"),
    ('price_filled', "FP32"),
    ('salesRank_Electronics', "FP32"),
    ('category_0_2_index', "INT32"),
    ('category_1_2_index', "INT32")
]

dtypes = {
    "INT32": np.int32,
    "INT64": np.int64,
    "FP32": np.float32
}

inputs = []
batch_size = 64
for column in columns:
    name = column[0]
    dtype = dtypes[column[1]]
    data = np.expand_dims(np.array(ratings.head(batch_size)[name], dtype=dtype), axis=-1)
    inputs.append(tritonhttpclient.InferInput(name, [batch_size, 1], column[1]))
    inputs[-1].set_data_from_numpy(data)

try:
    results = triton_client.infer(model_name, inputs, outputs=outputs).get_response()
    print("\nprediction results:\n", results["outputs"][0]["data"])
except Exception as e:
    print(f"Batch inference request failed: {e}")
    print("Ensure the Triton server is running and the model is loaded.")
    raise

## 3. Server Metrics

Not only can we scale serving our data, but we can also gather metrics on our model as well. This is crucial, finding the right metric to optimize for with recommender systems is not a trivial task. Check out this [great paper](https://www.kdd.org/exploration_files/19-1-Article3.pdf) explaining common pitfalls.

The short version is this:
* Recommender systems create a feedback loop between users and recommendations. Popular items train our models that these are good recommendations, thus serving them to more users and perpetuating the loop.
* Try to avoid metrics that are biased by human behavior. For instance, click through rate is one commonly used in the advertisement space, but if not careful, using this will train the model which position on a web page is popular as opposed to the content.

At the end of the day, the goal is to increase user engagement. Triton automatically serves usage metrics using [Prometheus](https://prometheus.io/). Copy and paste the URL (web address) for this notebook and set it to `my_url` below. Run the cell to see the metrics for our model. [Here](https://github.com/triton-inference-server/server/blob/main/docs/metrics.md) is a list of available metrics, but a good one to start with is `nv_inference_count` which displays how many predictions have been made.

In [ ]:
# PLATFORM-SPECIFIC: This cell requires the NVIDIA DLI platform.
# Replace the URL below with your DLI notebook URL to view Prometheus metrics.
# This will not work outside the DLI environment.

import IPython

my_url = "COPY_NOTEBOOK_URL"  # <-- Replace with your DLI notebook URL

if my_url == "COPY_NOTEBOOK_URL":
    import warnings
    warnings.warn(
        "Placeholder URL detected: 'my_url' is still set to 'COPY_NOTEBOOK_URL'. "
        "Please replace it with your actual DLI notebook URL before running this cell. "
        "You can find the URL in your browser's address bar."
    )
else:
    prometheus_url = my_url.rsplit(".com", 1)[0] + ".com:9090/graph"
    IPython.display.IFrame(prometheus_url, width=700, height=500)

## Wrap Up

We can take this a little further and hook these results into a service like [Grafana](https://grafana.com/) as explained in [this excellent blog post](https://blog.einstein.ai/benchmarking-tensorrt-inference-server/) by the SalesForce team, but for now, we have all the pieces to build an end-to-end recommender system.

Feeling ready? Head on over to [the next lab](3-04_assessment.ipynb) to put these new skills into action!